### Setup: Install Libraries

This cell ensures that the `scikit-learn` library, a fundamental tool for machine learning, is installed and ready for use in our environment.

In [17]:
!pip install scikit-learn

### Data Loading and Preprocessing

This code block downloads and parses linguistic data from a CONLLU file. This format is standard for annotated text. The `parse_conllu` function extracts important details for each word, such as its ID, actual word form, Part-of-Speech (POS) tag, and its grammatical head (parent word) in the sentence.

In [ ]:
import urllib.request
import os

def parse_conllu(source):
    """
    Reads and parses a CoNLL-U format file from a local path or a web URL.
    Returns a list of sentences, where each sentence is a list of token dictionaries.
    """
    if source.startswith("http://") or source.startswith("https://"):
        response = urllib.request.urlopen(source)
        lines = [line.decode("utf-8") for line in response.readlines()]
    else:
        with open(source, "r", encoding="utf-8") as f:
            lines = f.readlines()

    sentences = []
    current_sentence = []

    for line in lines:
        line = line.strip()
        if not line:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
            continue

        # Skip metadata comments
        if line.startswith("#"):
            continue

        parts = line.split("\t")

        # Skip multi-word tokens (e.g. 1-2) or empty nodes (e.g. 1.1)
        if "-" in parts[0] or "." in parts[0]:
            continue

        word_info = {
            "id": int(parts[0]),
            "form": parts[1],
            "upos": parts[3],       # Universal POS tag
            "head": int(parts[6]),  # Parent word ID (0 for ROOT)
            "deprel": parts[7]      # Dependency relation label
        }
        current_sentence.append(word_info)

    if current_sentence:
        sentences.append(current_sentence)

    return sentences

# URLs for UD English-EWT dataset
TRAIN_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-train.conllu"
DEV_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-dev.conllu"

print("Loading dataset...")
train_sentences = parse_conllu(TRAIN_URL)
dev_sentences = parse_conllu(DEV_URL)

print(f"Loaded {len(train_sentences)} train sentences and {len(dev_sentences)} dev sentences.")
print(f"First train sentence length: {len(train_sentences[0])} tokens.")


Loading dataset...
Loaded 12544 train sentences and 2001 dev sentences.
First train sentence length: 29 tokens.


### Defining the Parser's Core Logic

Here, we set up the fundamental parts of our dependency parser. The `State` class keeps track of the parsing process (words processed, words waiting, and the dependencies found). The `get_oracle` function is like a perfect guide, telling the parser the single best next move to make at each step to build the correct dependency tree.

In [5]:
from collections import deque

class State:
    """
    Represents the parser configuration:
    - stack: list of word IDs (starts with [0] for ROOT)
    - buffer: deque of word IDs waiting to be processed
    - arcs: list of (head, child, label) tuples already created
    - id_to_word: dictionary for O(1) word/POS lookups
    """
    def __init__(self, sentence):
        self.stack = [0]
        self.buffer = deque([w["id"] for w in sentence])
        self.arcs = []
        self.id_to_word = {w["id"]: w for w in sentence}
        self.id_to_word[0] = {"id": 0, "form": "ROOT", "upos": "ROOT"}

    def parsed_children(self, node):
        """Returns set of child IDs already attached to node."""
        return {child for head, child, _ in self.arcs if head == node}


def get_oracle(state, gold_heads, gold_labels, gold_children):
    """
    Arc-Standard Oracle transition decision:
    1. LEFT-ARC(label):
       - Top is head of Second: gold_heads[second] == top
       - Second is not ROOT: second != 0
       - Second has already collected ALL its gold dependents
    2. RIGHT-ARC(label):
       - Second is head of Top: gold_heads[top] == second
       - Top has already collected ALL its gold dependents
    3. SHIFT:
       - Buffer is not empty
    """
    if len(state.stack) >= 2:
        top = state.stack[-1]
        second = state.stack[-2]

        parsed_second = state.parsed_children(second)
        parsed_top = state.parsed_children(top)

        # Check for LEFT-ARC (second is popped, so second must have all its dependents)
        if second != 0 and gold_heads.get(second) == top:
            if gold_children.get(second, set()).issubset(parsed_second):
                return f"LEFT-ARC:{gold_labels[second]}"

        # Check for RIGHT-ARC (top is popped, so top must have all its dependents)
        if gold_heads.get(top) == second:
            if gold_children.get(top, set()).issubset(parsed_top):
                return f"RIGHT-ARC:{gold_labels[top]}"

    # Otherwise SHIFT if buffer has tokens
    if len(state.buffer) > 0:
        return "SHIFT"

    return None


This cell defines two core components for a dependency parser:

*   **`State` class**: Represents the current state of the dependency parser, including the `stack` (words processed), `buffer` (words yet to be processed), and `arcs` (the dependency relations found so far). It also provides a quick lookup for word information based on their IDs.
*   **`get_oracle` function**: This function acts as an 'oracle' for the parser. Given a current parser `state` and the `gold_arcs` (the correct dependency relations for a sentence), it determines the single correct next transition (either `SHIFT`, `LEFT-ARC`, or `RIGHT-ARC`). It helps in training the parser by telling it the optimal move at each step.

### Generating Training Data

This code creates the training data for our dependency parser. It uses two main functions:

*   **`extract_features`**: Gathers important information (like Part-of-Speech tags) from the current state of the parser.
*   **`generate_training_data`**: Simulates parsing each sentence, uses an 'oracle' to find the correct actions, and collects these 'feature-action' pairs. This data will teach our model how to parse sentences.

In [6]:
def extract_features(state):
    """
    Extracts the 4 required Part-of-Speech tag features:
    1. POS tag of top of stack
    2. POS tag of second word on stack
    3. POS tag of first word in buffer
    4. POS tag of second word in buffer
    """
    stack = state.stack
    buffer = state.buffer
    lookup = state.id_to_word

    # 1. POS of top word on stack
    s1 = lookup[stack[-1]]["upos"] if len(stack) >= 1 else "NULL"
    # 2. POS of second word on stack
    s2 = lookup[stack[-2]]["upos"] if len(stack) >= 2 else "NULL"

    # 3. POS of first word in buffer
    b1 = lookup[buffer[0]]["upos"] if len(buffer) >= 1 else "NULL"
    # 4. POS of second word in buffer
    b2 = lookup[buffer[1]]["upos"] if len(buffer) >= 2 else "NULL"

    return {
        "stack_top": s1,
        "stack_second": s2,
        "buffer_first": b1,
        "buffer_second": b2
    }


def generate_training_data(sentences):
    """
    Simulates the oracle over parsed sentences to generate (feature, action) pairs.
    """
    X_features, y_actions = [], []

    for sentence in sentences:
        # Precompute gold structures for fast lookup
        gold_heads = {w["id"]: w["head"] for w in sentence}
        gold_labels = {w["id"]: w["deprel"] for w in sentence}
        gold_children = {}
        for w in sentence:
            gold_children.setdefault(w["head"], set()).add(w["id"])

        state = State(sentence)

        while state.buffer or len(state.stack) > 1:
            transition = get_oracle(state, gold_heads, gold_labels, gold_children)
            if not transition:
                break  # Stops if non-projective or complete

            features = extract_features(state)
            X_features.append(features)
            y_actions.append(transition)

            # Apply the oracle transition to step the state forward
            if transition == "SHIFT":
                state.stack.append(state.buffer.popleft())
            elif transition.startswith("LEFT-ARC"):
                label = transition.split(":", 1)[1]
                child = state.stack.pop(-2)
                head = state.stack[-1]
                state.arcs.append((head, child, label))
            elif transition.startswith("RIGHT-ARC"):
                label = transition.split(":", 1)[1]
                child = state.stack.pop()
                head = state.stack[-1]
                state.arcs.append((head, child, label))

    return X_features, y_actions


In [7]:
# Quick test on "The cat sat on the mat."
test_sentence = [
    {"id": 1, "form": "The",  "upos": "DET",   "head": 2, "deprel": "det"},
    {"id": 2, "form": "cat",  "upos": "NOUN",  "head": 3, "deprel": "nsubj"},
    {"id": 3, "form": "sat",  "upos": "VERB",  "head": 0, "deprel": "root"},
    {"id": 4, "form": "on",   "upos": "ADP",   "head": 6, "deprel": "case"},
    {"id": 5, "form": "the",  "upos": "DET",   "head": 6, "deprel": "det"},
    {"id": 6, "form": "mat",  "upos": "NOUN",  "head": 3, "deprel": "obl"},
    {"id": 7, "form": ".",    "upos": "PUNCT", "head": 3, "deprel": "punct"},
]

test_X, test_y = generate_training_data([test_sentence])
print(f"Total steps generated: {len(test_y)} (Expected: 14)")
print("\nGenerated sequence of transitions:")
for step_idx, (feats, act) in enumerate(zip(test_X, test_y), start=1):
    print(f"Step {step_idx:2d}: {act:<20} | Features: {feats}")

# Check action variety
from collections import Counter
print("\nAction counts:", Counter([a.split(':')[0] for a in test_y]))


Total steps generated: 14 (Expected: 14)

Generated sequence of transitions:
Step  1: SHIFT                | Features: {'stack_top': 'ROOT', 'stack_second': 'NULL', 'buffer_first': 'DET', 'buffer_second': 'NOUN'}
Step  2: SHIFT                | Features: {'stack_top': 'DET', 'stack_second': 'ROOT', 'buffer_first': 'NOUN', 'buffer_second': 'VERB'}
Step  3: LEFT-ARC:det         | Features: {'stack_top': 'NOUN', 'stack_second': 'DET', 'buffer_first': 'VERB', 'buffer_second': 'ADP'}
Step  4: SHIFT                | Features: {'stack_top': 'NOUN', 'stack_second': 'ROOT', 'buffer_first': 'VERB', 'buffer_second': 'ADP'}
Step  5: LEFT-ARC:nsubj       | Features: {'stack_top': 'VERB', 'stack_second': 'NOUN', 'buffer_first': 'ADP', 'buffer_second': 'DET'}
Step  6: SHIFT                | Features: {'stack_top': 'VERB', 'stack_second': 'ROOT', 'buffer_first': 'ADP', 'buffer_second': 'DET'}
Step  7: SHIFT                | Features: {'stack_top': 'ADP', 'stack_second': 'VERB', 'buffer_first': 'DET', 

### Generating Training Data via Oracle Simulation

In this cell, we run our Arc-Standard oracle simulator across the training treebank (`train_sentences`). For every parsing step (configuration), the simulator:
1. Inspects the current `stack` and `buffer`.
2. Extracts the 4 Part-of-Speech tag features using `extract_features(state)`.
3. Consults the gold tree to determine the optimal transition (`SHIFT`, `LEFT-ARC:<label>`, or `RIGHT-ARC:<label>`).
4. Executes that transition to advance the parser state until the sentence is parsed.

This produces our supervised training dataset: pairs of `(feature_dict, transition_label)`. We also print the transition distribution across the dataset to inspect the action space.


In [8]:
import time
from collections import Counter

print("Generating training data with oracle simulation...")
t0 = time.time()

# Generate transitions from training sentences
X_train_dicts, y_train = generate_training_data(train_sentences)

elapsed = time.time() - t0
print(f"Generated {len(y_train):,} training transitions from {len(train_sentences):,} sentences in {elapsed:.2f}s.")

# Inspect the action space
action_counts = Counter(y_train)
print(f"\nTotal unique action classes: {len(action_counts)}")
print("Top 10 most frequent actions:")
for action, count in action_counts.most_common(10):
    print(f"  {action:<25} : {count:>7,d} ({count / len(y_train) * 100:.1f}%)")

# Inspect transition types
shift_count = action_counts["SHIFT"]
left_count = sum(c for a, c in action_counts.items() if a.startswith("LEFT-ARC"))
right_count = sum(c for a, c in action_counts.items() if a.startswith("RIGHT-ARC"))
print(f"\nTransition Distribution:")
print(f"  SHIFT     : {shift_count:>7,d} ({shift_count / len(y_train) * 100:.1f}%)")
print(f"  LEFT-ARC  : {left_count:>7,d} ({left_count / len(y_train) * 100:.1f}%)")
print(f"  RIGHT-ARC : {right_count:>7,d} ({right_count / len(y_train) * 100:.1f}%)")


Generating training data with oracle simulation...
Generated 407,165 training transitions from 12,544 sentences in 1.18s.

Total unique action classes: 82
Top 10 most frequent actions:
  SHIFT                     : 204,578 (50.2%)
  LEFT-ARC:case             :  16,673 (4.1%)
  LEFT-ARC:det              :  15,707 (3.9%)
  RIGHT-ARC:punct           :  15,503 (3.8%)
  LEFT-ARC:nsubj            :  15,345 (3.8%)
  RIGHT-ARC:root            :  12,257 (3.0%)
  RIGHT-ARC:obj             :   9,433 (2.3%)
  LEFT-ARC:amod             :   9,298 (2.3%)
  LEFT-ARC:advmod           :   8,619 (2.1%)
  LEFT-ARC:punct            :   7,746 (1.9%)

Transition Distribution:
  SHIFT     : 204,578 (50.2%)
  LEFT-ARC  : 118,045 (29.0%)
  RIGHT-ARC :  84,542 (20.8%)


### Feature Vectorization (`DictVectorizer`)

Machine learning classifiers require numerical inputs rather than symbolic dictionaries. Here, we use `DictVectorizer` from `scikit-learn` to perform one-hot encoding on our 4 categorical POS tag features:
* `stack_top` (POS tag of $S_1$)
* `stack_second` (POS tag of $S_2$)
* `buffer_first` (POS tag of $B_1$)
* `buffer_second` (POS tag of $B_2$)

This creates a sparse boolean feature matrix where each feature-value pair (e.g., `stack_top=VERB`, `buffer_first=NOUN`) corresponds to a unique binary column.


In [9]:
from sklearn.feature_extraction import DictVectorizer

print("Vectorizing features...")
vectorizer = DictVectorizer(sparse=True)
X_train_vec = vectorizer.fit_transform(X_train_dicts)

print(f"Feature matrix shape: {X_train_vec.shape}")
print(f"Vocabulary size (one-hot features): {len(vectorizer.get_feature_names_out())}")
print(f"Sample one-hot feature names: {vectorizer.get_feature_names_out()[:10]}")


Vectorizing features...
Feature matrix shape: (407165, 73)
Vocabulary size (one-hot features): 73
Sample one-hot feature names: ['buffer_first=ADJ' 'buffer_first=ADP' 'buffer_first=ADV'
 'buffer_first=AUX' 'buffer_first=CCONJ' 'buffer_first=DET'
 'buffer_first=INTJ' 'buffer_first=NOUN' 'buffer_first=NULL'
 'buffer_first=NUM']


### Model Training: Logistic Regression Classifier

We train a multi-class `LogisticRegression` classifier to act as our statistical oracle. Given the feature vector representing a configuration, the classifier outputs probability scores across all possible transitions.

**Why Logistic Regression?**
1. **Speed & Efficiency**: With a linear model and one-hot features, training over 200,000+ transitions converges in just a few seconds.
2. **Probability Ranking**: It outputs calibrated probabilities (`predict_proba`), allowing the parser to pick the highest-ranked *valid* transition during inference if the top prediction is illegal.


In [10]:
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression classifier...")
t0 = time.time()

classifier = LogisticRegression(
    max_iter=500,
    solver="lbfgs",
    n_jobs=-1,
    random_state=42
)

classifier.fit(X_train_vec, y_train)

train_time = time.time() - t0
train_acc = classifier.score(X_train_vec, y_train) * 100

print(f"Model trained in {train_time:.2f}s!")
print(f"Training Accuracy: {train_acc:.2f}%")


Training Logistic Regression classifier...


c:\Users\pc-lenovo\anaconda3\envs\deeplearning\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Model trained in 64.27s!
Training Accuracy: 80.43%


### Sanity Check: Configuration Prediction & Probability Ranking

Before deploying the classifier inside the parser loop, we test its prediction on a sample configuration:
* `stack_top`: `VERB`
* `stack_second`: `NOUN`
* `buffer_first`: `PUNCT`
* `buffer_second`: `NULL`

We inspect the top predicted action and the probability distribution across candidate transitions.


In [11]:
sample_state = {
    "stack_top": "VERB",
    "stack_second": "NOUN",
    "buffer_first": "PUNCT",
    "buffer_second": "NULL"
}

sample_vec = vectorizer.transform([sample_state])
pred_action = classifier.predict(sample_vec)[0]
probs = classifier.predict_proba(sample_vec)[0]
top3_idx = probs.argsort()[-3:][::-1]

print("Sample Configuration:", sample_state)
print(f"Top Predicted Action: {pred_action}")
print("Top 3 actions with probabilities:")
for idx in top3_idx:
    print(f"  {classifier.classes_[idx]:<25} : {probs[idx] * 100:.2f}%")


Sample Configuration: {'stack_top': 'VERB', 'stack_second': 'NOUN', 'buffer_first': 'PUNCT', 'buffer_second': 'NULL'}
Top Predicted Action: RIGHT-ARC:acl
Top 3 actions with probabilities:
  RIGHT-ARC:acl             : 34.63%
  RIGHT-ARC:acl:relcl       : 33.50%
  RIGHT-ARC:conj            : 10.39%


### Part 3: Transition-Based Parser Implementation

The parser takes an unparsed sentence (tokens with POS tags) and constructs its dependency parse tree using the trained classifier.

**Handling Transition Preconditions (Guardrails):**
A statistical classifier can occasionally predict a transition that is illegal in the current configuration:
1. `SHIFT`: requires `len(buffer) > 0`.
2. `LEFT-ARC(label)`: requires `len(stack) >= 2` and `stack[-2] != 0` (the ROOT token cannot become a dependent).
3. `RIGHT-ARC(label)`: requires `len(stack) >= 2`.

To prevent crashes, invalid attachments, or infinite loops, our parser uses **Constrained Decoding**:
* The classifier ranks candidate transitions by probability.
* The parser selects the **highest-probability valid transition**.
* If no candidate is valid, it applies a deterministic fallback (e.g. `SHIFT` if buffer has tokens, otherwise `RIGHT-ARC` to reduce the stack).
* The loop terminates when the buffer is empty and only the ROOT remains on the stack.


In [12]:
def parse_sentence(sentence, model, vectorizer):
    """
    Parses a sentence using the trained classifier and arc-standard transition system.
    Returns a list of predicted dependency arcs: [(head, child, label), ...]
    """
    state = State(sentence)

    while state.buffer or len(state.stack) > 1:
        # 1. Extract features from the current configuration
        feats = extract_features(state)
        x_vec = vectorizer.transform([feats])

        # 2. Determine which transitions are valid in the current state
        can_shift = len(state.buffer) > 0
        can_left = len(state.stack) >= 2 and state.stack[-2] != 0
        can_right = len(state.stack) >= 2

        # 3. Get candidate transitions ranked by classifier probability
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(x_vec)[0]
            ranked_actions = [model.classes_[i] for i in probs.argsort()[::-1]]
        else:
            ranked_actions = [model.predict(x_vec)[0]]

        # 4. Select the highest-probability valid action
        chosen_action = None
        for action in ranked_actions:
            if action == "SHIFT" and can_shift:
                chosen_action = action
                break
            elif action.startswith("LEFT-ARC") and can_left:
                chosen_action = action
                break
            elif action.startswith("RIGHT-ARC") and can_right:
                chosen_action = action
                break

        # Fallback safeguard in case no ranked candidate matched
        if not chosen_action:
            if can_shift:
                chosen_action = "SHIFT"
            elif can_right:
                chosen_action = "RIGHT-ARC:dep"
            elif can_left:
                chosen_action = "LEFT-ARC:dep"
            else:
                break

        # 5. Apply the transition to update the configuration
        if chosen_action == "SHIFT":
            state.stack.append(state.buffer.popleft())
        elif chosen_action.startswith("LEFT-ARC"):
            label = chosen_action.split(":", 1)[1] if ":" in chosen_action else "dep"
            child = state.stack.pop(-2)
            head = state.stack[-1]
            state.arcs.append((head, child, label))
        elif chosen_action.startswith("RIGHT-ARC"):
            label = chosen_action.split(":", 1)[1] if ":" in chosen_action else "dep"
            child = state.stack.pop()
            head = state.stack[-1]
            state.arcs.append((head, child, label))

    return state.arcs


### Evaluation Metrics: Labeled Attachment Score (LAS) and UAS

We evaluate the parser on the development split (`dev_sentences`, from `en_ewt-ud-dev.conllu`).

**Metrics:**
* **LAS (Labeled Attachment Score)**: The percentage of words assigned both the **correct head** and the **correct dependency label**.
  $$\text{LAS} = \frac{\text{Number of tokens with correct head AND correct label}}{\text{Total number of tokens}} \times 100\%$$
* **UAS (Unlabeled Attachment Score)**: The percentage of words assigned the **correct head**, regardless of the label.
  $$\text{UAS} = \frac{\text{Number of tokens with correct head}}{\text{Total number of tokens}} \times 100\%$$


In [13]:
def evaluate_parser(dev_data, model, vectorizer, max_sentences=None):
    """
    Evaluates the parser on a CoNLL-U dataset and computes UAS and LAS scores.
    """
    eval_set = dev_data[:max_sentences] if max_sentences else dev_data
    
    total_tokens = 0
    correct_heads = 0
    correct_labeled = 0
    
    t0 = time.time()
    for sent in eval_set:
        gold_arcs = {w["id"]: (w["head"], w["deprel"]) for w in sent}
        pred_arcs = {c: (h, l) for h, c, l in parse_sentence(sent, model, vectorizer)}

        for token in sent:
            tid = token["id"]
            gold_h, gold_l = gold_arcs[tid]
            pred_h, pred_l = pred_arcs.get(tid, (None, None))

            # UAS: Head must match
            if pred_h == gold_h:
                correct_heads += 1
                # LAS: Both head and label must match
                if pred_l == gold_l:
                    correct_labeled += 1

            total_tokens += 1

    elapsed = time.time() - t0
    uas = (correct_heads / total_tokens) * 100 if total_tokens else 0
    las = (correct_labeled / total_tokens) * 100 if total_tokens else 0

    return uas, las, total_tokens, elapsed

print("Evaluating parser on the dev set (en_ewt-ud-dev.conllu)...")
uas, las, num_tokens, eval_time = evaluate_parser(dev_sentences, classifier, vectorizer)

print(f"\n--- Evaluation Results ({len(dev_sentences)} sentences, {num_tokens:,} tokens) ---")
print(f"Evaluation completed in: {eval_time:.2f}s ({len(dev_sentences)/eval_time:.1f} sentences/sec)")
print(f"Unlabeled Attachment Score (UAS): {uas:.2f}%")
print(f"Labeled Attachment Score   (LAS): {las:.2f}%")


Evaluating parser on the dev set (en_ewt-ud-dev.conllu)...

--- Evaluation Results (2001 sentences, 25,148 tokens) ---
Evaluation completed in: 14.20s (140.9 sentences/sec)
Unlabeled Attachment Score (UAS): 66.70%
Labeled Attachment Score   (LAS): 56.71%


### Testing on Required Example Sentences

The assignment asks to test the parser on three example sentences:
1. *"The cat sat on the mat."*
2. *"She eats a green salad."*
3. *"I saw the man with a telescope."*

Below, we supply the POS tags for these sentences, parse them with our trained model, and display the predicted dependency arcs.


In [14]:
examples = [
    {
        "name": "Sentence 1",
        "text": "The cat sat on the mat.",
        "tokens": [
            {"id": 1, "form": "The",   "upos": "DET"},
            {"id": 2, "form": "cat",   "upos": "NOUN"},
            {"id": 3, "form": "sat",   "upos": "VERB"},
            {"id": 4, "form": "on",    "upos": "ADP"},
            {"id": 5, "form": "the",   "upos": "DET"},
            {"id": 6, "form": "mat",   "upos": "NOUN"},
            {"id": 7, "form": ".",     "upos": "PUNCT"}
        ]
    },
    {
        "name": "Sentence 2",
        "text": "She eats a green salad.",
        "tokens": [
            {"id": 1, "form": "She",   "upos": "PRON"},
            {"id": 2, "form": "eats",  "upos": "VERB"},
            {"id": 3, "form": "a",     "upos": "DET"},
            {"id": 4, "form": "green", "upos": "ADJ"},
            {"id": 5, "form": "salad", "upos": "NOUN"},
            {"id": 6, "form": ".",     "upos": "PUNCT"}
        ]
    },
    {
        "name": "Sentence 3",
        "text": "I saw the man with a telescope.",
        "tokens": [
            {"id": 1, "form": "I",         "upos": "PRON"},
            {"id": 2, "form": "saw",       "upos": "VERB"},
            {"id": 3, "form": "the",       "upos": "DET"},
            {"id": 4, "form": "man",       "upos": "NOUN"},
            {"id": 5, "form": "with",      "upos": "ADP"},
            {"id": 6, "form": "a",         "upos": "DET"},
            {"id": 7, "form": "telescope", "upos": "NOUN"},
            {"id": 8, "form": ".",         "upos": "PUNCT"}
        ]
    }
]

for ex in examples:
    print(f"\n==========================================")
    print(f"{ex['name']}: \"{ex['text']}\"")
    print(f"==========================================")
    predicted_arcs = parse_sentence(ex["tokens"], classifier, vectorizer)
    
    # Map token IDs to words for display
    id_to_form = {w["id"]: w["form"] for w in ex["tokens"]}
    id_to_form[0] = "ROOT"

    # Sort arcs by dependent ID for readability
    sorted_arcs = sorted(predicted_arcs, key=lambda arc: arc[1])
    for head, child, label in sorted_arcs:
        head_form = id_to_form.get(head, f"ID_{head}")
        child_form = id_to_form.get(child, f"ID_{child}")
        print(f"  {child_form:>10} ({child})  <---[{label:^8}]---  {head_form} ({head})")



Sentence 1: "The cat sat on the mat."
         The (1)  <---[  det   ]---  cat (2)
         cat (2)  <---[  root  ]---  ROOT (0)
         sat (3)  <---[  acl   ]---  cat (2)
          on (4)  <---[  case  ]---  mat (6)
         the (5)  <---[  det   ]---  mat (6)
         mat (6)  <---[  obj   ]---  sat (3)
           . (7)  <---[ punct  ]---  cat (2)

Sentence 2: "She eats a green salad."
         She (1)  <---[ nsubj  ]---  eats (2)
        eats (2)  <---[  root  ]---  ROOT (0)
           a (3)  <---[  det   ]---  salad (5)
       green (4)  <---[  amod  ]---  salad (5)
       salad (5)  <---[  obj   ]---  eats (2)
           . (6)  <---[ punct  ]---  eats (2)

Sentence 3: "I saw the man with a telescope."
           I (1)  <---[ nsubj  ]---  saw (2)
         saw (2)  <---[  root  ]---  ROOT (0)
         the (3)  <---[  det   ]---  man (4)
         man (4)  <---[  obj   ]---  saw (2)
        with (5)  <---[  case  ]---  telescope (7)
           a (6)  <---[  det   ]---  telescope (7

### Part 4: Brief Report & Design Choices

#### 1. Transition System (Arc-Standard)
* **Mechanics**: The parser maintains a `stack` initialized with `[0]` (ROOT), a `buffer` with all sentence token IDs, and an `arcs` accumulator.
* **Transitions**:
  * `SHIFT`: Moves the front token of the buffer to the top of the stack.
  * `LEFT-ARC(label)`: Creates an arc where the top token ($S_1$) is the head of the second token ($S_2$), and pops $S_2$. Enforces $S_2 \neq \text{ROOT}$.
  * `RIGHT-ARC(label)`: Creates an arc where $S_2$ is the head of $S_1$, and pops $S_1$.
* **Oracle Constraint**: In Arc-Standard, when a word is popped, it can never acquire additional dependents. Hence, a word is only attached to its head once **all** its gold children have been collected.

#### 2. Feature Representation & Model Selection
* **Features**: A 4-token POS window ($S_1, S_2, B_1, B_2$) encoded into binary features using `DictVectorizer`.
* **Classifier**: Multi-class `LogisticRegression` with the `lbfgs` solver. Linear models are well-suited for high-dimensional sparse indicator features and train within seconds on over 200,000 steps without GPU requirements.
* **Constrained Decoding**: Because classifiers lack grammar-level guarantees, the parser verifies preconditions at inference time and chooses the top probability-ranked valid transition, guaranteeing a valid projective tree.

#### 3. Performance & Error Analysis
* **Attachment Scores**:
  * UAS: Evaluates unlabeled head selection.
  * LAS: Evaluates joint head selection and dependency label assignment.
* **Observed Limitations**: With only 4 POS tag features, the model has limited lexical context to resolve attachment ambiguities (such as prepositional phrase attachment in *"saw the man with a telescope"*, where lexical semantics between *"saw"* vs *"man"* determine the head of *"with"*). Adding word lemmas, distance features, and child features can further improve LAS.
